# Präambel

In [1]:
from sympy import *
from IPython.display import display, Math, Latex, Markdown
#init_session()
init_printing(use_unicode=True)

In [2]:
class dotdict(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

def md(s):
    display(Markdown(s))
    
def ausnutzung(p, stab=None):
    color = '#000'
    emoji = ''
    if p > 1:
        color = '#f00'
        emoji = '🔴'
    elif p > 0.7:
        color = '#0f0'
        emoji = '🟢'
    elif p < 0.7:
        color = '#00f'
        emoji = '🟡'
    #md(f'#### <span style="color:{color};">{int(p*100)}%</span>')
    qs = stab.querschnitt if stab else ''
    md(f'#### {emoji} {int(N(p, 6)*100)}% ({qs})')
    return p


def note(s):
    md(f'<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">{s}</span></div>')
#ausnutzung(13.37)
#ausnutzung(0.238)
N(0.238, 2) * 100

In [3]:
%run querschnitte.ipynb

In [115]:
from dataclasses import dataclass
from decimal import Decimal
from functools import partial

@dataclass
class Schnittgroeszen:
    N: float = None
    V: float = 0
    Vz: float = 0
    My: float = 0
    Mz: float = 0
    ψ: float = None
    ψz: float = None
    
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    M = lambda self, achse: self.um_achse('M', achse)


    
@dataclass
class Stahlguete:
    guete: str
    E: int
    fy: float
    fu: float
    
@dataclass
class Stab(Querschnitt, Stahlguete):
    l: float
    sg: dict
    Lcry: float = None
    Lcrz: float = None
    

    def __init__(self, guete, QS, **kwargs):
        sg = kwargs.pop('sg', None)
        if isinstance(sg, dict):
            self.sg = sg
        else:
            self.sg = {None: sg}
            
        for k,v in kwargs.items():
            self.__setattr__(k, v)
            
        Querschnitt.__init__(self, **QS.__dict__)
        Stahlguete.__init__(self, **guete.__dict__)
        
    def um_achse(self, attr, achse):
        return self.__getattribute__(f'{attr}{achse}')
        
    Lcr = lambda self, achse: self.um_achse('Lcr', achse)

S235 = Stahlguete(guete='S235', E=21000, fy=23.5, fu=36)
S355 = Stahlguete(guete='S355', E=21000, fy=35.5, fu=49)


# Nachweis-Funktionen

## Querschnitte

In [5]:
#import inspect
#inspect.getsourcelines(vglsp)


### Vergleichsspannung

In [92]:
def Sy(qs):
    Sf = (qs.b * qs.tf) * (qs.h/2 - qs.tf/2)
    Sw = ((qs.h/2 - qs.tf) * qs.tw) * (qs.h/2 - qs.tf)/2
    #display(f'Sf = {Sf}   Sw = {Sw}')
    return Sf + Sw

def vglsp(stab, sg=None, debug=True):
    last = stab.sg[sg]
    σx = last.N/stab.A
    if last.My:
        σx += (last.My*100)/stab.W('y')
    τ = -(last.V * Sy(stab))/(stab.I('y') * stab.tw) if last.V else 0
    #display(f'sigmax = {N(σx,5)}  tau = {N(τ,5)}   S = {N(Sy(stab),5)} W = {N(stab.W('y'), 5)}')
    σv = sqrt(σx**2 + 3 * τ**2)
    if debug:
        print(last.N)
        print(last.V)
        print(last.My*100)
        print(stab.A)
        print(stab.W('y'))
        print(N(Sy(stab), 5))
        print(N(σx,5))
        print(N(τ,5))
        print(N(σv, 5))
        print('')
        print(N(σv/stab.fy, 5))
    return ausnutzung(N(σv/stab.fy, 5), stab)


def dbg(val):
    print(N(val,5)) if val is not None else print('')

### Legacy

In [93]:
### Zug [EN 6.2.3]
def zug(stab, sg=None, A_net=None):
    γ_M0 = 1
    γ_M2 = 1.25
    N_plRd = (stab.A * stab.fy)/γ_M0
    N_uRd = (0.9 * A_net * stab.fu)/γ_M2 if A_net else oo
    last = stab.sg[sg]

    return ausnutzung(N(last.N/min(N_plRd, N_uRd), 3), stab)
    
### Druck [EN 6.2.4]
def druck(stab, sg=None):
    γ_M0 = 1
    N_cRd = (stab.A * stab.fy)/γ_M0
    last = stab.sg[sg]
    
    return ausnutzung(N(last.N/N_cRd, 3), stab)
    
### Biegung [EN 6.2.5]
def biegung(um_achse, stab, sg=None):
    γ_M0 = 1

    M_plRd = (stab.Wpl(um_achse) * stab.fy)/γ_M0
    
    #h = stab.h if um_achse == 'y' else stab.b
    #W_elmin = stab.I(um_achse)/(h/2)
    #M_elRd = (W_elmin * stab.fy)/γ_M0
    
    last = stab.sg[sg]
    return ausnutzung(N(last.M(um_achse)*100/M_plRd, 3), stab)

biegung_y = partial(biegung, 'y')
biegung_z = partial(biegung, 'z')

### Querkraft [EN 6.2.6] TODO: E-E
def querkraft(um_achse, stab, sg=None):
    γ_M0 = 1
    V_plRd = (stab.Av(um_achse) * (stab.fy/sqrt(3)) )/ γ_M0

    last = stab.sg[sg]
    return ausnutzung(last.V/V_plRd, stab)

querkraft_y = partial(querkraft, 'y')
querkraft_z = partial(querkraft, 'z')

### Kombi: M + N + V [EN 6.2.10]
def kombi_MNV(um_achse, stab, sg=None, debug=False):
    assert(um_achse == 'y'), 'Nur fuer My implementiert'
    γ_M0 = 1
    last = stab.sg[sg]
    
    V_plRd = (stab.Av(um_achse) * (stab.fy/sqrt(3)) )/ γ_M0
    assert((last.V / V_plRd) <= 0.5), 'Abminderung aufgrund V_Ed notwendig!'
    
    N_plRd = (stab.A * stab.fy)/γ_M0 
    M_NRd = M_plRd = (stab.Wpl(um_achse) * stab.fy)/γ_M0
    if um_achse == 'y' and ( (last.N > 0.25 * N_plRd) or (last.N > ((0.5 * stab.hw * stab.tw * stab.fy) / γ_M0)) ):
        n = last.N/N_plRd
        a = min(0.5, (stab.A - 2 * stab.b * stab.tf)/stab.A)
        M_NRd = min(M_plRd, M_plRd * (1-n)/(1-0.5*a))
        if debug == True:
            display(Math(f'n = {N(n, 5)}, a = {N(a, 5)}, M_NRd = {N(M_NRd, 5)}'))
    else:
        if debug == True:
            display(Math(f'M_NRd = {N(M_NRd, 5)}'))
    return ausnutzung(last.M(um_achse)*100/M_NRd, stab)

kombi_MNV_y = partial(kombi_MNV,'y')
kombi_MNV_z = partial(kombi_MNV,'z')

## Bauteil

In [94]:
γ_M1 = 1 #TODO tripple-check

def λ1(stab):
    return pi * sqrt(stab.E/stab.fy)

def λyz(um_achse, stab, debug=False):
    if debug:
        display(f'lcr = {stab.Lcr(um_achse)} i = {stab.i(um_achse)}  lamba_1 = {N(λ1(stab))}')
    return stab.Lcr(um_achse)/(stab.i(um_achse) * λ1(stab))

def KL2α(KL: str):
    return {'a0': 0.13, 'a': 0.21, 'b': 0.34, 'c': 0.49, 'd': 0.76}.get(KL)


def Phi(um_achse, stab):
    λ = λyz(um_achse, stab)
    α = KL2α(stab.KL(um_achse))
    Φ = 0.5* (1 + α * (λ-0.2) + λ**2)
    return Φ
    
    
def χyz(um_achse, stab, debug=False):
    λ = λyz(um_achse, stab)
    Φ = Phi(um_achse, stab)
    chi = min(1, 1/(Φ + sqrt(Φ**2 - λ**2)))
    #if debug:
    #    display(f'λ = {N(λ)}  α = {N(α)}  Φ = {N(Φ)}  χ = {N(chi)}')
    return chi

### Biegeknicken [B 8.7.6.1]

In [84]:
def biege_knicken(um_achse, stab, sg=None, debug=True):
    last = stab.sg[sg]
    assert(stab.Lcr(um_achse))
    assert(last.ψ is not None or last.My == 0)

    ## Widerstaende
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) # NB nur W_y relevant, egal um welche Achse
    My_Rd =  W_y * (stab.fy / γ_M1)

    ## Beiwerte
    n_y = last.N / (χyz('y', stab) * N_Rd)
    k_y = min((1 + (λyz('y', stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    α = 0.6 if um_achse == 'z' else 1
    C_my = max(0.6 + 0.4 * (last.ψ or 0),
               0.4)
    #if debug:
    #    display(Math(f'N_Rd = {N(N_Rd,5)}  My_Rd = {N(My_Rd,5)} \\, W_y = {N(W_y,3)}'))
    #    display(Math(f'n_y = {N(n_y,5)}  k_y = {N(k_y,5)} \\, C_my = {N(C_my,3)}'))
    #    display(Math(f'χ = {N(χyz(um_achse, stab, debug=True), 5)}'))

    ## Nachweis
    res = (last.N/(χyz(um_achse, stab) * N_Rd)) + (α * k_y * (C_my * last.My*100)/My_Rd)

    if debug:
        print(last.N)
        print(last.My*100)
        dbg(last.ψ)
        dbg(k_y)
        dbg(C_my)
        dbg(λyz(um_achse, stab))
        dbg(Phi(um_achse, stab))
        dbg(χyz('y', stab))
        dbg(χyz('z', stab))
        print(N_Rd)
        print(My_Rd)
        print('')
        dbg(res)
        
    return ausnutzung(N(res, 5), stab)

biege_knicken_y = partial(biege_knicken, 'y')
biege_knicken_z = partial(biege_knicken, 'z')

### Biegedrillknicken [B 8.7.6.1]

In [105]:
def lambda_LT(stab):
    λz = λyz('z', stab)
    
    k_c = 1 # Auf der sicheren Seite
    k_p = 0.9 * 1 / (1 + (1/20) * (λz/(stab.h/stab.tf))**2)**0.25
    return k_c * k_p * λz

def Phi_LT(stab):
    λLT = lambda_LT(stab)
    λLT0 = 0.4  # Höchstwert (EC3 Seite 70)
    β = 0.75    # Mindestwert
    αLT = KL2α(stab.KL_BDK)
    
    return 0.5 * ( 1 + αLT * (λLT - λLT0) + β * λLT**2)
    

def χLT(stab, debug=False): ## Gem. EC3 (6.3.2.3)
    debug = False
    λLT = lambda_LT(stab)
    β = 0.75    # Mindestwert
    
    ΦLT = Phi_LT(stab)
    
    return min(1 / (ΦLT + sqrt(ΦLT**2 - β * λLT**2)),
               1,
               1 / (λLT**2),
              )

In [106]:
def BDK(um_achse, stab, sg=None, debug=True):
    last = stab.sg[sg]
    assert(stab.Lcr(um_achse))
    assert(last.ψ is not None)
    
    N_Rd = stab.A * (stab.fy/γ_M1)
    W_y = stab.Iy/(stab.h * 0.5) ## CHECK past?
    My_Rd =  W_y * (stab.fy / γ_M1) # W_y passt?


    
    ## Beiwerte
    C_my = max(0.4, (0.6 + 0.4 * last.ψ))
    C_MLT = C_my
    
    n_y = last.N / (χyz('y', stab) * N_Rd)
    k_y = min((1 + (λyz('y', stab) - 0.2) * n_y),
              (1 + 0.8 * n_y),
             )
    
    n_z = last.N / (χyz('z', stab) * N_Rd)
    λz = λyz('z', stab)
    assert (λz >= 0.4)
    k_LT = max(1 - (0.1 * n_z * λz)/(C_MLT - 0.25),
               1 - (0.1 * n_z     )/(C_MLT - 0.25),
              )


    interaktionsbeiwert = {'y': k_y * C_my,
                           'z': k_LT,
                          }.get(um_achse)

    res = (last.N/(χyz(um_achse, stab) * N_Rd)) + (interaktionsbeiwert * last.My*100/(χLT(stab, debug=debug) * My_Rd))
    
    if debug:
        print(last.N)
        print(last.My*100)
        dbg(last.ψ)
        dbg(k_y)
        dbg(k_LT)
        dbg(C_my)
        dbg(C_MLT)
        dbg(λyz(um_achse, stab))
        dbg(lambda_LT(stab))
        dbg(Phi_LT(stab))
        dbg(χyz('y', stab))
        dbg(χyz('z', stab))
        dbg(χLT(stab))
        print(N_Rd)
        print(My_Rd)
        print('')
        dbg(res)
        
    return ausnutzung(N(res, 5), stab)

BDK_y = partial(BDK, 'y')
BDK_z = partial(BDK, 'z')

# Stäbe

In [12]:
staebe = dotdict()

## Pfette

In [86]:
staebe.pfette = Stab(
    S235, HEB180,
    l=675, Lcry=675, Lcrz=675,
    sg = {
        'druck': Schnittgroeszen(
            N  = 102,    
            My = 70.21, 
            V = 42.52, 
            ψ  = 0.95),
        'zug': Schnittgroeszen(
            N  = 69.1,   
            My = 70.21,
            V = 42.52,
            ψ  = 0.95),
        },
)
#done 2025-04-19 8:50

### Bauteil

In [85]:
biege_knicken_y(staebe.pfette, sg='druck', debug=True)
biege_knicken_z(staebe.pfette, sg='druck', debug=True)

102
7020.999999999999
0.95000
1.0772
0.98000
0.93832
1.0657
0.63652
0.29208
1533.375
10003.166666666668

0.84542


#### 🟢 84% (HEB 180)

102
7020.999999999999
0.95000
1.0772
0.98000
1.5728
2.0731
0.63652
0.29208
1533.375
10003.166666666668

0.67230


#### 🟡 67% (HEB 180)

### Querschnitt

In [15]:
#zug(staebe.pfette, sg='zug')
#kombi_MNV_y(staebe.pfette, sg='zug')
#kombi_MNV_y(staebe.pfette, sg='druck')
#querkraft_y(staebe.pfette, sg='druck')

In [16]:
vglsp(staebe.pfette, sg='druck')
vglsp(staebe.pfette, sg='zug')

102
42.52
7020.999999999999
65.25
425.6666666666667
233.71
18.057
-3.0517
18.815

0.80064


#### 🟢 80% (HEB 180)

69.1
42.52
7020.999999999999
65.25
425.6666666666667
233.71
17.553
-3.0517
18.332

0.78007


#### 🟢 78% (HEB 180)

## OG

In [17]:
staebe.obergurt = Stab(
    S235, HEB320,
    l=250.3, Lcry=250.3, Lcrz=250.3,
    sg = {
        'zug': Schnittgroeszen(
            N  = 2090.88,
            My = 118.01,
            Mz = 30.51,
            V  = 44.59,
            Vz = 13.29,
            ψ  = 28.74/118.01,
            ψz = 9.71/25.38,
        ),
        'druck': Schnittgroeszen(
            N  = 317.92,
            #My = 14.69,
            My = 118.01,
            Mz = 15.08,
            #V  = 7.01,
            V  = 44.59,
            Vz = 6.44,
            ψz = 3.84/14.85, # anderer verlauf, aber maximum
            ψ  = 4.07/14.69,
        ),
    })
#done 2025-04-19 8:50

### Querschnitt

In [18]:
#zug(staebe.obergurt, sg='zug')
#kombi_MNV_y(staebe.obergurt, sg='zug')
##kombi_MNV_z(staebe.obergurt, sg='zug')
##TODO zug um z!! (?) 
#
#kombi_MNV_y(staebe.obergurt, sg='druck')

In [121]:
vglsp(staebe.obergurt, sg='zug')

2090.88
44.59
11801.0
161.3
1926.25
1032.9
19.089
-1.2994
19.221

0.81793


#### 🟢 81% (HEB 320)

### Bauteil

In [87]:
biege_knicken_y(staebe.obergurt, sg='druck')
biege_knicken_z(staebe.obergurt, sg='druck')

317.92
11801.0
0.27706
0.99940
0.71082
0.19285
0.51738
1.0000
0.92239
3790.55
45266.875

0.26907


#### 🟡 26% (HEB 320)

317.92
11801.0
0.27706
0.99940
0.71082
0.35208
0.59924
1.0000
0.92239
3790.55
45266.875

0.20205


#### 🟡 20% (HEB 320)

## UG1 Feld

In [21]:
staebe.untergurt_feld = Stab(
    S235, HEB320,
    l=0, Lcry=240, Lcrz=378.1,
    sg = {
        'druck': Schnittgroeszen(
            N  = 2532.81,
            My = 58.98,
            V  = 42.88,
            ψ  = 1
        ),
        'zug': Schnittgroeszen(
            N  = 384.45,
            #My = 8.85,
            My = 58.98,
            #V  = 7.21,
            V  = 42.88,
            ψ  = 1,
        ),
    })
#done 2025-04-19 8:56


### Querschnitt

In [22]:
#kombi_MNV_y(staebe.untergurt_feld, sg='zug-y')
#kombi_MNV_y(staebe.untergurt_feld, sg='druck-y')
#druck(staebe.untergurt_feld, sg='druck-y')

In [23]:
vglsp(staebe.untergurt_feld, sg='zug',debug=True)
vglsp(staebe.untergurt_feld, sg='druck',debug=True)

384.45
42.88
5898.0
161.3
1926.25
1032.9
5.4454
-1.2496
5.8597

0.24935


#### 🟡 24% (HEB 320)

2532.81
42.88
5898.0
161.3
1926.25
1032.9
18.764
-1.2496
18.889

0.80378


#### 🟢 80% (HEB 320)

### Bauteil

In [107]:
BDK_y(staebe.untergurt_feld, sg='druck', debug=True)

2532.81
5898.0
1.0000
0.98992
0.94257
1.0000
1.0000
0.18492
0.47865
0.59929
1.0000
0.82501
0.96891
3790.55
45266.875

0.80131


#### 🟢 80% (HEB 320)

In [108]:
BDK_z(staebe.untergurt_feld, sg='druck')

2532.81
5898.0
1.0000
0.98992
0.94257
1.0000
1.0000
0.53185
0.47865
0.59929
1.0000
0.82501
0.96891
3790.55
45266.875

0.93667


#### 🟢 93% (HEB 320)

## UG2 UG3 Kragarm

In [66]:
staebe.untergurt_krag_li = Stab(
    S235, HEB320,
    l=256, Lcry=256, Lcrz=648.7,
    #l=256, Lcry=256, Lcrz=3.95,
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 1867.9,
            My = 34.38,
            V  = 25.95,
            ψ  = 27/34,
        ),
        'druck-z': Schnittgroeszen(
            N  = 1867.9,
            My = 34.38,
            V  = 25.95,
            ψ  = 0,
        ),
        'zug-y': Schnittgroeszen(
            N  = 188.58,
            #My = 3.63,
            #V  = 3.04,
            My = 34.38,
            V  = 25.95,
            ψ  = 2.2/3.1, 
        ),
        'zug-z': Schnittgroeszen(
            N  = 188.58,
            My = 34.38,
            V  = 25.95,
            #My = 3.63,
            #V  = 3.04,
            ψ  = 0,
        ),
    })
# ^^^ done 2025-04-19 8:06

staebe.untergurt_krag_re = Stab(
    S235, HEB320,
    l=0, Lcry=253, Lcrz=2*485.8,
    sg = {
        'druck-y': Schnittgroeszen(
            N  = 453.44,
            My = 14.61,
            V  = 6.45,
            ψ  = 3.32/14.64,
        ),
        'druck-z': Schnittgroeszen(
            N  = 453.44,
            My = 14.61,
            V  = 6.45,
            ψ  = 0
        ),
    })
# ^^^ done 2025-04-19 09:11

note('zug UG krag re nicht relevant, weil schon mit UG krag li abgedeckt')
note('ψ werte fuer -z stab doublechecken!!')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">zug UG krag re nicht relevant, weil schon mit UG krag li abgedeckt</span></div>

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">ψ werte fuer -z stab doublechecken!!</span></div>

### Querschnitt

In [67]:
#zug(staebe.untergurt_krag_li, sg='zug-y')
#kombi_MNV_y(staebe.untergurt_krag_li, sg='zug-y')
#kombi_MNV_y(staebe.untergurt_krag_li, sg='druck-y')
#kombi_MNV_y(staebe.untergurt_krag_re, sg='druck-y')

In [68]:
vglsp(staebe.untergurt_krag_li, sg='zug-y')
vglsp(staebe.untergurt_krag_li, sg='druck-y')
vglsp(staebe.untergurt_krag_re, sg='druck-y')

188.58
25.95
3438.0000000000005
161.3
1926.25
1032.9
2.9539
-0.75622
3.2313

0.13750


#### 🟡 13% (HEB 320)

1867.9
25.95
3438.0000000000005
161.3
1926.25
1032.9
13.365
-0.75622
13.429

0.57145


#### 🟡 57% (HEB 320)

453.44
6.45
1461.0
161.3
1926.25
1032.9
3.5696
-0.18796
3.5844

0.15253


#### 🟡 15% (HEB 320)

### Bauteil

In [110]:
BDK_y(staebe.untergurt_krag_li, sg='druck-y')
BDK_z(staebe.untergurt_krag_li, sg='druck-z')


1867.9
3438.0000000000005
0.79412
0.99864
0.88627
0.91765
0.91765
0.19725
0.82120
0.82449
1.0000
0.59218
0.80539
3790.55
45266.875

0.57920


#### 🟡 57% (HEB 320)

1867.9
3438.0000000000005
0
0.99864
0.78305
0.60000
0.60000
0.91248
0.82120
0.82449
1.0000
0.59218
0.80539
3790.55
45266.875

0.90598


#### 🟢 90% (HEB 320)

In [111]:
BDK_y(staebe.untergurt_krag_re, sg='druck-y')
BDK_z(staebe.untergurt_krag_re, sg='druck-z')

453.44
1461.0
0.22678
0.99939
0.92499
0.69071
0.69071
0.19493
1.2299
1.2083
1.0000
0.36184
0.56215
3790.55
45266.875

0.15926


#### 🟡 15% (HEB 320)

453.44
1461.0
0
0.99939
0.90554
0.60000
0.60000
1.3667
1.2299
1.2083
1.0000
0.36184
0.56215
3790.55
45266.875

0.38259


#### 🟡 38% (HEB 320)

## Streben FW

In [31]:
staebe.strebe = Stab(
    S235, HEA160,
    l=192, Lcry=192, Lcrz=192,
    sg = {
        'druck': Schnittgroeszen(
            N  = 632.36,
            My = 0,
            V  = 0,
        ),
        'zug': Schnittgroeszen(
            N  = 766.91,
            My = 0,
            V  = 0,
        ),
    }
)
# ^^^ done 2025-04-19 09:17


### Querschnitt (Zug)

In [32]:
#zug(staebe.strebe, sg='zug')
vglsp(staebe.strebe, sg='zug')
vglsp(staebe.strebe, sg='druck')

766.91
0
0
38.77
220.13157894736844
116.43
19.781
0
19.781

0.84175


#### 🟢 84% (HEA 160)

632.36
0
0
38.77
220.13157894736844
116.43
16.311
0
16.311

0.69407


#### 🟡 69% (HEA 160)

### Bauteil

In [95]:
biege_knicken_y(staebe.strebe, sg='druck')
biege_knicken_z(staebe.strebe, sg='druck')

632.36
0

1.0804
0.60000
0.31118
0.56732
0.95999
0.83531
911.095
5173.092105263158

0.72299


#### 🟢 72% (HEA 160)

632.36
0

1.0804
0.60000
0.51368
0.70879
0.95999
0.83531
911.095
5173.092105263158

0.83091


#### 🟢 83% (HEA 160)

## Streben UG

In [34]:
staebe.strebe_UG = Stab(
    S235, HEB180, l=149, Lcry=149, Lcrz=149,
    sg = {
        'druck': Schnittgroeszen(
            N  = 1233.37,
            My = 0,
            V  = 0,
        ),
        'zug': Schnittgroeszen(
            N  = 193.97, 
            My = 0,
            V  = 0,
        ),
    }
)
# ^^^ done 2025-04-19 9:24

### Querschnitt

In [35]:
#zug(staebe.strebe_UG, sg='zug')
#druck(staebe.strebe_UG, sg='druck')
#vglsp(staebe.strebe_UG, sg='zug')
vglsp(staebe.strebe_UG, sg='druck')

1233.37
0
0
65.25
425.6666666666667
233.71
18.902
0
18.902

0.80435


#### 🟢 80% (HEB 180)

### Bauteil

In [96]:
biege_knicken_y(staebe.strebe_UG, sg='druck')
biege_knicken_z(staebe.strebe_UG, sg='druck')

1233.37
0

1.0057
0.60000
0.20712
0.52266
0.99748
0.92493
1533.375
10003.166666666668

0.80639


#### 🟢 80% (HEB 180)

1233.37
0

1.0057
0.60000
0.34717
0.59632
0.99748
0.92493
1533.375
10003.166666666668

0.86963


#### 🟢 86% (HEB 180)

## Knickpfette

In [37]:
staebe.knickpfette = Stab(
    S235, HEA100, l=675, Lcry=675, Lcrz=675,
    sg = {
        'druck': Schnittgroeszen(
            N  = 25.07,
            My = 1.3,
            V  = 0.8,
            ψ  = 0.95,
        ),
        #'zug': Schnittgroeszen(
        #    N  = 0 
        #    My = 0,
        #    V  = 0,
        #),
    }
)
# ^^^ done 2025-04-19 9:28

In [113]:
#druck(staebe.knickpfette, sg='druck')
#kombi_MNV_y(staebe.knickpfette, sg='druck')
vglsp(staebe.knickpfette, sg='druck')

BDK_y(staebe.knickpfette, sg='druck')
BDK_z(staebe.knickpfette, sg='druck')
#biege_knicken_y(staebe.knickpfette, sg='druck')
#biege_knicken_z(staebe.knickpfette, sg='druck')

25.07
0.8
130.0
21.24
72.75
39.200
2.9673
-0.17961
2.9835

0.12696


#### 🟡 12% (HEA 100)

25.07
130.0
0.95000
1.1549
0.93357
0.98000
0.98000
1.7703
2.5754
3.3570
0.25941
0.10356
0.15077
499.14
1709.625

0.76443


#### 🟢 76% (HEA 100)

25.07
130.0
0.95000
1.1549
0.93357
0.98000
0.98000
2.8635
2.5754
3.3570
0.25941
0.10356
0.15077
499.14
1709.625

0.95581


#### 🟢 95% (HEA 100)

## Dachverband

In [39]:
staebe.dachverband = Stab(
    S235, RD30, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 150.48, 
        My = 0,
        V  = 0,
    ),
)
# ^^^ done 2025-04-19 9:33

### Querschnitt

In [40]:
vglsp(staebe.dachverband)

150.48
0
0
7.07
0.0
0
21.284
0
21.284

0.90571


#### 🟢 90% (RD 30)

## WAV Wandverband

In [119]:
staebe.wandverband = Stab(
    S355, RD45, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 354.59,
        My = 0,
        V  = 0,
    ),
)

### Querschnitt

In [120]:
#zug(staebe.wandverband)
vglsp(staebe.wandverband)

354.59
0
0
15.9
0.0
0
22.301
0
22.301

0.62820


#### 🟡 62% (RD 45)

## Knickverband

In [43]:
staebe.knickverband = Stab(
    S235, RD20, l=0, Lcry=0, Lcrz=0,
    sg = Schnittgroeszen(
        N  = 64.24,
        My = 0,
        V  = 0,
    ),
)
# ^^^ done 2025-04-19 9:39


### Querschnitt

In [44]:
vglsp(staebe.knickverband)

64.24
0
0
3.14
0.0
0
20.459
0
20.459

0.87058


#### 🟢 87% (RD 20)

## 7 Stütze A

In [45]:
staebe.stuetzeA = Stab(
    S235, HEA160, l=650, Lcry=650, Lcrz=650,
    sg = {
        'zug': Schnittgroeszen(
            N  = 692.89,# + (-141.98),
            My = 0,
            V  = 0,
        ),
        'druck': Schnittgroeszen(
            N  = 113.45 + (97.76),
            My = 0,
            V  = 0,
        ),
    })

# ^^^ done 2025-04-19 9:47
note('dachverband einfluss weggelassen, auf sicheren seite (eh nicht maßgebend)')
note('HEA140 waren 93%, aber keine schraubenloecher geplant ansich... fuer spaetere adaptionen HEA160?')

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">dachverband einfluss weggelassen, auf sicheren seite (eh nicht maßgebend)</span></div>

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">HEA140 waren 93%, aber keine schraubenloecher geplant ansich... fuer spaetere adaptionen HEA160?</span></div>

### Querschnitt

In [46]:
#zug(staebe.stuetzeA, sg='zug')
#druck(staebe.stuetzeA, sg='druck')
vglsp(staebe.stuetzeA, sg='zug')
vglsp(staebe.stuetzeA, sg='druck')

692.89
0
0
38.77
220.13157894736844
116.43
17.872
0
17.872

0.76050


#### 🟢 76% (HEA 160)

211.21
0
0
38.77
220.13157894736844
116.43
5.4478
0
5.4478

0.23182


#### 🟡 23% (HEA 160)

### Bauteil

In [97]:
biege_knicken_y(staebe.stuetzeA, sg='druck')
biege_knicken_z(staebe.stuetzeA, sg='druck')

211.21
0

1.3291
0.60000
1.0535
1.2000
0.56351
0.24830
911.095
5173.092105263158

0.41139


#### 🟡 41% (HEA 160)

211.21
0

1.3291
0.60000
1.7390
2.3892
0.56351
0.24830
911.095
5173.092105263158

0.93363


#### 🟢 93% (HEA 160)

## COB Stütze B

In [48]:
staebe.stuetzeB = Stab(
    S235, HEM650, l=775, Lcry=0, Lcrz=625,
    sg = {
        'druck-max': Schnittgroeszen(
            N  = 1332.1,
            My = 1044.13,
            V  = 696.43,
            ψ  = 1,
        ),
        'druck-maxM': Schnittgroeszen(
            N  = 1332.1,
            My = 1044.13,
            V  = 696.43,
            ψ  = 1,
        ),
        'druck-stabenden': Schnittgroeszen(
            N  = 1332.1,
            My = 1044.13,
            V  = 696.43,
            ψ  = 1,
        ),
        'zug': Schnittgroeszen(
            N  = 167.46,
            My = 235.99,
            V  = 157.33,
        ),
    })

### Knicklänge (um y)

In [49]:
h_Fachwerk = 108                 # Achsabstand der Gurte in Mitte des Feldbereichs vom Fachwerk
z = h_Fachwerk * 0.5
l_R = 500
l = 775
l1 = staebe.stuetzeA.l
N_A = abs(-staebe.stuetzeA.sg['zug'].N)
#N_A = -staebe.stuetzeA.sg['zug'].N
N_B = abs(staebe.stuetzeB.sg['druck-max'].N)
F = N_A * (l / l1)   # <--- Einsparungspotential!  ggf hoehe bis traegermitte waehlbar


display(f'NA = {N_A}')
display(f'NB = {N_B}')
note('auf sicheren Seite mit absolutwerten gerechnet')

EI = staebe.stuetzeB.E * staebe.stuetzeB.Iy
EI_R = staebe.obergurt.E * ((staebe.obergurt.Iy + staebe.obergurt.A * z**2) + (staebe.untergurt_feld.Iy + staebe.untergurt_feld.A * z**2))
ν = 1 / ( (l_R * EI)/(l * EI_R) + 1)
η = ( (3 * ν) / ( (0.216 * ν**2 + 1) * N_B + F)) * (EI/l**2)
print(η )
staebe.stuetzeB.Lcry = N(pi * sqrt(EI/(η*N_B)))
staebe.stuetzeB.Lcry

'NA = 692.89'

'NB = 1332.1'

<div style="width=100%; display: flex; justify-content: end"><span style="background-color:lightgreen; padding: 0.2rem 0.4rem; margin-right: 3rem; text-align: right; font-weight: bold;">auf sicheren Seite mit absolutwerten gerechnet</span></div>

10.578697177595629


### Querschnitt

In [50]:
#zug(staebe.stuetzeB, sg='zug')
#druck(staebe.stuetzeB, sg='druck-y')
#querkraft_y(staebe.stuetzeB, sg='druck-y')
#kombi_MNV_y(staebe.stuetzeB, sg='zug')
#kombi_MNV_y(staebe.stuetzeB, sg='druck-y')
vglsp(staebe.stuetzeB, sg='zug')
vglsp(staebe.stuetzeB, sg='druck-max')

167.46
157.33
23599.0
373.7
8434.131736526946
4738.4
3.2461
-1.2602
3.9117

0.16646


#### 🟡 16% (HEM 650)

1332.1
696.43
104413.00000000001
373.7
8434.131736526946
4738.4
15.944
-5.5783
18.643

0.79334


#### 🟢 79% (HEM 650)

### Bauteil

In [112]:
BDK_y(staebe.stuetzeB, sg='druck-max')

1332.1
104413.00000000001
1.0000
1.1116
0.97049
1.0000
1.0000
0.78959
0.84002
0.87242
0.80160
0.63969
0.73857
8781.949999999999
198202.09580838322

0.98208


#### 🟢 98% (HEM 650)

In [98]:
biege_knicken_z(staebe.stuetzeB, sg='druck-max')

1332.1
104413.00000000001
1.0000
1.1116
1.0000
0.93339
1.0603
0.80160
0.63969
8781.949999999999
198202.09580838322

0.58847


#### 🟡 58% (HEM 650)